# Beyond Pace: Pace-Driven Running vs Navigation-Driven Sports
### *Why Speed Tells Only Part of the Story*

**Tools:** Python (pandas, matplotlib, seaborn, sklearn) / Power BI  
**Data:** Garmin Connect exports / 4 athlete profiles / ~2012 records

---

## Project Idea

Standard running metrics — pace, speed, distance — work well for road runners. 

But what about athletes who navigate forests, read maps, and make split-second route decisions?

This project compares **two navigation-driven athlete profiles** with **two pace-driven runners** to show that pace alone is insufficient to evaluate performance in navigation-driven sports.

> *"Profiles, not people"* — each dataset represents an activity profile, not an individual.

---
## What is navigation-driven sports

This are a unique category of athletic activities where a participant's success depends equally on their physical endurance and their ability to actively find their way through an environment. Instead of following a fixed, clearly marked track athletes must use tools like maps, compasses or even radio direction finding apparatus to navigate through diverse wooded terrain to locate specific checkpoints.

In this research we are talking about 2 types of Navigation-Driven Sports: Orienteering and ARDF.

---

## Research Questions

1. How does the distribution of activity distances differ between navigation-driven and pace-driven athletes?

2. How does average pace differ between athlete profiles?

3. How does average heart rate differ across navigation-driven athletes and a pace-driven runners?

4. How does elevation gain differ across athlete profiles — and what does it reveal about terrain and decision-making?

5. Which metrics are most strongly correlated in activity data across different athlete types?

6. Does higher elevation gain slow down pace — and does this differ by athlete profile?

7. Is pace alone sufficient to evaluate athletic performance — what does Pace vs HR reveal?

8. How does training volume change over time — is there seasonal patterns in navigation-driven sports?

9. How consistent is pace across athlete profiles — and what does variability reveal about activity type?

10. Can heart rate be predicted from pace and elevation — and what does model quality tell us about navigation-driven sports?



---

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

C:\Users\viver\AppData\Local\Temp\ipykernel_24208\1850576127.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# завантажуємо датасети

df_personal = pd.read_csv("../data/raw/garmin_personal.csv")
df_partner = pd.read_csv("../data/raw/garmin_partner.csv")
df_activity_bob = pd.read_csv("../data/raw/activity_bob.csv")
df_activity_log = pd.read_csv("../data/raw/activity_log.csv")

### ЗНАЙОМСТВО З ДАНИМИ

In [3]:
# загальна інформація
print("ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_personal:")
print()
df_personal.info()

# базова статистика
print()
print("БАЗОВА СТАТИСТИКА ПО df_personal:")
print()
stats = df_personal.describe()
stats.round(2)

ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_personal:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307 entries, 0 to 306
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Activity Type           307 non-null    object 
 1   Date                    307 non-null    object 
 2   Favorite                307 non-null    bool   
 3   Title                   307 non-null    object 
 4   Distance                307 non-null    float64
 5   Calories                307 non-null    object 
 6   Time                    307 non-null    object 
 7   Avg HR                  307 non-null    int64  
 8   Max HR                  307 non-null    int64  
 9   Aerobic TE              307 non-null    object 
 10  Avg Run Cadence         307 non-null    int64  
 11  Max Run Cadence         307 non-null    int64  
 12  Avg Pace                307 non-null    object 
 13  Best Pace               307 non-null    object 
 14  Tota

,Distance,Avg HR,Max HR,Avg Run Cadence,Max Run Cadence,Avg Stride Length,Training Stress Score®,Number of Laps
count,307.00,307.00,307.00,307.00,307.00,307.00,307.0,307.00
mean,8.03,145.81,175.43,156.89,214.57,0.91,0.0,6.64
std,12.08,18.63,14.95,17.85,32.64,2.65,0.0,13.26
min,0.00,67.00,72.00,104.00,124.00,0.00,0.0,1.00
25%,3.70,140.00,171.00,146.00,186.00,0.56,0.0,1.00
50%,6.91,150.00,179.00,159.00,231.00,0.69,0.0,1.00
75%,9.74,157.50,184.00,173.00,243.00,0.79,0.0,10.00
max,146.35,174.00,204.00,180.00,253.00,38.07,0.0,154.00


In [4]:
df_personal.head()

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Avg Stride Length,Training Stress Score®,Steps,Decompression,Best Lap Time,Number of Laps,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Trail Running,2026-05-17 11:56:00,False,"🦊3,5 training",8.38,479,01:06:42,151,187,4.0,...,0.76,0.0,"10,334",No,00:01:45.1,11,01:05:22,01:25:19,164,214
1,Trail Running,2026-05-16 14:20:24,False,"🦊3,5 training",9.64,607,01:49:50,139,183,3.1,...,0.61,0.0,"11,556",No,00:01:13.5,13,01:38:19,01:49:50,114,200
2,Trail Running,2026-05-09 11:45:01,False,🦊 144 training,10.90,740,02:31:34,130,175,3.1,...,0.50,0.0,"13,758",No,00:02:56.8,14,01:59:11,02:31:34,121,186
3,Trail Running,2026-05-03 10:16:03,False,Lutskyi raion Running,4.53,299,00:49:57,136,174,2.8,...,0.64,0.0,"5,446",No,00:06:07.4,5,00:43:48,00:49:57,211,250
4,Trail Running,2026-05-02 12:04:59,False,"🦊ARDF, ЧУ 3,5, Луцьк",11.35,617,01:54:51,141,186,3.8,...,0.63,0.0,"12,616",No,00:05:13.9,12,01:33:28,01:54:51,179,218


In [5]:
# загальна інформація
print("ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_partner:")
print()
df_partner.info()

# базова статистика
print()
print("БАЗОВА СТАТИСТИКА ПО df_partner:")
print()
stats = df_partner.describe()
stats.round(2)

ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_partner:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Activity Type           280 non-null    object 
 1   Date                    280 non-null    object 
 2   Favorite                280 non-null    bool   
 3   Title                   280 non-null    object 
 4   Distance                280 non-null    float64
 5   Calories                280 non-null    object 
 6   Time                    280 non-null    object 
 7   Avg HR                  280 non-null    int64  
 8   Max HR                  280 non-null    int64  
 9   Avg Run Cadence         280 non-null    object 
 10  Max Run Cadence         280 non-null    object 
 11  Avg Pace                280 non-null    object 
 12  Best Pace               280 non-null    object 
 13  Total Ascent            280 non-null    object 
 14  Total

,Distance,Avg HR,Max HR,Training Stress Score®,Number of Laps
count,280.00,280.00,280.00,280.0,280.00
mean,4.61,148.21,172.04,0.0,5.24
std,3.40,20.90,23.32,0.0,3.44
min,0.00,73.00,92.00,0.0,1.00
25%,1.92,139.00,161.00,0.0,3.00
50%,4.03,155.00,179.00,0.0,5.00
75%,6.86,163.00,187.00,0.0,7.00
max,23.91,175.00,222.00,0.0,25.00


In [6]:
df_partner.head()

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Avg Run Cadence,...,Avg Stride Length,Training Stress Score®,Steps,Decompression,Best Lap Time,Number of Laps,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Running,2026-04-19 12:30:22,False,Kyievo-Sviatoshynskyi raion Running,2.60,232,00:25:27,158,181,119,...,0.87,0.0,"3,460",No,00:05:41.6,3,00:22:33,00:25:27,157,190
1,Running,2026-04-19 11:45:37,False,Kyievo-Sviatoshynskyi raion Running,4.96,410,00:41:47,160,180,132,...,0.90,0.0,"6,346",No,00:07:08.9,5,00:36:49,00:41:47,158,191
2,Running,2026-04-18 13:15:35,False,Kyiv Running,2.42,171,00:17:55,159,190,148,...,0.91,0.0,"2,892",No,00:02:21.9,3,00:16:20,00:17:55,124,158
3,Running,2026-04-14 16:56:03,False,Kyievo-Sviatoshynskyi raion Running,1.48,165,00:21:59,140,169,82,...,0.82,0.0,"2,198",No,00:06:54.2,2,00:15:09,00:21:59,176,192
4,Running,2026-04-14 16:05:04,False,Kyievo-Sviatoshynskyi raion Running,3.98,391,00:41:57,157,183,102,...,0.93,0.0,"5,334",No,00:07:55.0,4,00:32:40,00:41:57,158,192


In [7]:
# загальна інформація
print("ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_activity_bob:")
print()
df_activity_bob.info()

# базова статистика
print()
print("БАЗОВА СТАТИСТИКА ПО df_activity_bob:")
print()
stats = df_activity_bob.describe()
stats.round(2)

ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_activity_bob:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Activity Type             736 non-null    object
 1   Date                      736 non-null    object
 2   Favorite                  736 non-null    bool  
 3   Title                     736 non-null    object
 4   Distance                  736 non-null    object
 5   Calories                  736 non-null    object
 6   Time                      736 non-null    object
 7   Avg HR                    736 non-null    object
 8   Max HR                    736 non-null    object
 9   Aerobic TE                736 non-null    object
 10  Avg Run Cadence           736 non-null    object
 11  Max Run Cadence           736 non-null    object
 12  Avg Pace                  736 non-null    object
 13  Best Pace                 736 non-null

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Body Battery Drain,Min Temp,Decompression,Best Lap Time,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation
count,736,736,736,736,736,736,736,736,736,736,...,736,736,736,736,736,736,736,736,736,736
unique,10,733,2,42,508,506,698,69,87,47,...,36,27,1,173,35,27,310,377,79,91
top,Running,2015-11-23 08:06:20,False,Untitled,"10,00",--,01:15:00,--,--,--,...,--,--,No,--:--:--,--,--,--:--:--,00:NaN:NaN,--,--
freq,523,2,729,251,8,82,5,397,397,479,...,602,692,736,490,403,687,425,351,296,296


In [8]:
df_activity_bob.head()

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Body Battery Drain,Min Temp,Decompression,Best Lap Time,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Running,2026-05-19 07:15:28,False,Vyshhorodskyi raion - Пороговое значен,"10,32",663,02:02:14,103,156,"2,7",...,-23,--,No,00:02:00,6,--,02:01:22,02:02:14,119,134
1,Running,2026-05-16 10:30:56,False,Vyshhorodskyi raion - Длит. проб.,"11,26",781,02:13:34,112,190,"3,3",...,-32,--,No,01:05:34,2,--,02:10:30,02:13:34,116,133
2,Running,2026-05-10 08:18:24,False,Vyshhorodskyi raion - База,"10,32",734,01:47:45,153,182,"3,1",...,-26,--,No,00:27:00,2,--,01:46:54,01:47:45,118,134
3,Running,2026-05-09 06:22:41,False,Vyshhorodskyi raion - База,"10,73",722,01:55:54,142,158,"3,0",...,-25,--,No,00:40:00,2,--,01:54:55,01:55:54,118,133
4,Running,2026-05-07 06:00:01,False,Vyshhorodskyi raion - Пороговое значен,"9,67",644,01:44:42,103,175,"2,9",...,-18,--,No,00:02:00,6,--,01:42:26,01:44:42,118,133


In [9]:
# загальна інформація
print("ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_activity_log:")
print()
df_activity_log.info()

# базова статистика
print()
print("БАЗОВА СТАТИСТИКА ПО df_activity_log:")
print()
stats = df_activity_log.describe()
stats.round(2)

ЗАГАЛЬНА ІНФОРМАЦІЯ ПРО df_activity_log:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 689 entries, 0 to 688
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Activity Type      689 non-null    object 
 1   Date               689 non-null    object 
 2   Title              689 non-null    object 
 3   Distance           689 non-null    float64
 4   Calories           689 non-null    object 
 5   Time               689 non-null    object 
 6   Avg HR             689 non-null    object 
 7   Max HR             689 non-null    object 
 8   Avg Run Cadence    689 non-null    object 
 9   Max Run Cadence    689 non-null    object 
 10  Avg Pace           689 non-null    object 
 11  Best Pace          689 non-null    object 
 12  Elev Gain          689 non-null    object 
 13  Elev Loss          689 non-null    object 
 14  Avg Stride Length  689 non-null    float64
 15  Best Lap Time      689 non-null 

,Distance,Avg Stride Length,Number of Laps
count,689.00,689.00,689.00
mean,5.16,1.28,5.58
std,3.94,0.34,4.01
min,0.00,0.00,1.00
25%,2.07,1.23,2.00
50%,4.03,1.29,5.00
75%,7.96,1.36,8.00
max,18.01,1.86,19.00


___

### ПОРІВНЯННЯ КОЛОНОК ДАТАФРЕЙМІВ

In [10]:
# зручний вивід для наочного порівняння:

df = pd.DataFrame({
    "garmin_personal": pd.Series(df_personal.columns.tolist()),
    "garmin_partner": pd.Series(df_partner.columns.tolist()),
    "activity_bob": pd.Series(df_activity_bob.columns.tolist()),
    "activity_log": pd.Series(df_activity_log.columns.tolist())
})
df

,garmin_personal,garmin_partner,activity_bob,activity_log
0,Activity Type,Activity Type,Activity Type,Activity Type
1,Date,Date,Date,Date
2,Favorite,Favorite,Favorite,Title
3,Title,Title,Title,Distance
4,Distance,Distance,Distance,Calories
5,Calories,Calories,Calories,Time
6,Time,Time,Time,Avg HR
7,Avg HR,Avg HR,Avg HR,Max HR
8,Max HR,Max HR,Max HR,Avg Run Cadence
9,Aerobic TE,Avg Run Cadence,Aerobic TE,Max Run Cadence


___

### АНАЛІЗ ПРОПУЩЕНИХ ЗНАЧЕНЬ

In [11]:
# Аналіз пропущених значень

for name, df in [
    ("garmin_personal", df_personal),
    ("garmin_partner", df_partner),
    ("activity_bob", df_activity_bob),
    ("activity_log", df_activity_log)
]:
    missing = df.isnull().sum()
    missing_percent = (missing / len(df) * 100).round(1)
    print(f"\n{name}")
    print(missing_percent[missing_percent > 0])





garmin_personal
Series([], dtype: float64)

garmin_partner
Series([], dtype: float64)

activity_bob
Series([], dtype: float64)

activity_log
Series([], dtype: float64)


___

### CLEANING

#### Одразу позбудемось пропущених значень:

In [12]:
df_personal = df_personal.replace('--', None)
df_partner = df_partner.replace('--', None)
df_activity_bob = df_activity_bob.replace('--', None)
df_activity_log = df_activity_log.replace('--', None)

####  Залишаємо тільки потрібні типи активностей:

In [13]:
keep_types = ["Running", "Trail Running", "Treadmill Running", "Indoor Running", "Other"]

df_personal = df_personal[df_personal["Activity Type"].isin(keep_types)]
df_partner = df_partner[df_partner["Activity Type"].isin(keep_types)]
df_activity_bob = df_activity_bob[df_activity_bob["Activity Type"].isin(keep_types)]
df_activity_log = df_activity_log[df_activity_log["Activity Type"].isin(keep_types)]

___
#### Перевіримо на наявність аномалій:

In [14]:
# activity_log має нестандартний формат часу,
# конвертуємо його аби обробити аномалії одразу з 4 датасетів:

def time_to_hours(t):
    if pd.isna(t):
        return None
    t = str(t)
    parts = t.split(":")
    if len(parts) == 3:
        return int(parts[0]) + int(parts[1]) / 60 + float(parts[2]) / 3600
    elif len(parts) == 2:
        return float(parts[0]) / 60 + float(parts[1]) / 3600
    return None

У функції використовуємо 2 константи:

speed_threshold=20 км/год — обираємо максимальний темп бігу навіть для елітних атлетів,
тому будь-яке значення вище є GPS-артефактом, а не реальною активністю.

steps_threshold=500 — обираємо умовний поріг - вдвічі менший за норму ~1000-1400 крок/км,
щоб не відфільтрувати ходьбу, орієнтування на повільних ділянках.

Записи з аномальною швидкістю і малою кількістю кроків — GPS-шум, не реальний рух:

In [15]:
# функція для виявлення аномальних записів на основі швидкості та кількості кроків/км:
# отримаємо df з аномальними записами:

def detect_anomalies(df, name, speed_threshold=20, steps_theshold=500, use_custom_time=False):
    df["Distance"] = pd.to_numeric(df["Distance"].astype(str).str.replace(",", "."), errors="coerce") # конвертація перед обрахунком
    
    # окремо для activity_log з нестандартним форматом:
    if use_custom_time:
        df["Time_hours"] = df["Time"].apply(time_to_hours)
    else:
        df["Time_hours"] = pd.to_timedelta(df["Time"]).dt.total_seconds() / 3600
    
    df["speed_kmh"] = df["Distance"] / df["Time_hours"]

    if "Steps" in df.columns:
        df["Steps"] = pd.to_numeric(
            df["Steps"].astype(str).str.replace(",", "").str.replace(".", "", regex=False),
            errors="coerce"
        )
        df["steps_per_km"] = df["Steps"] / df["Distance"]
        anomalies = df[
            (df["speed_kmh"] > speed_threshold) &
            (
                (df["steps_per_km"] < steps_theshold) | 
                (df["steps_per_km"].isna())         # якщо NaN кроків при аномальній швидкості
            )
        ]
    else:
        anomalies = df[df["speed_kmh"] > speed_threshold]
    
    print(f"{name}: found {len(anomalies)} anomalies")
    return anomalies

# Засстосуємо для всіх 4 датасетів:
anomalies_personal = detect_anomalies(df_personal, "personal")
anomalies_partner = detect_anomalies(df_partner, "partner")
anomalies_bob = detect_anomalies(df_activity_bob, "bob")
anomalies_log = detect_anomalies(df_activity_log, "log", use_custom_time=True)

personal: found 4 anomalies
partner: found 2 anomalies
bob: found 2 anomalies
log: found 0 anomalies


In [16]:
# виведем знайдені аномалії:

for name, anomalies in [
    ("personal", anomalies_personal),
    ("partner", anomalies_partner),
    ("bob", anomalies_bob),
    ("log", anomalies_log)
]:
    print(f"\n=== {name} ===")
    cols = ["Title", "Distance", "speed_kmh"]
    if "steps_per_km" in anomalies.columns:
        cols.append("steps_per_km")
    display(anomalies[cols].head(10))


=== personal ===


,Title,Distance,speed_kmh,steps_per_km
17,🤣🦊ARDF 144 ЧУ Вінниця - running during an air ...,94.51,46.742135,131.266533
18,Vinnytsia - running during an air raid,111.04,173.650738,42.903458
28,running during an air raid,30.57,201.634298,22.898266
53,Kyiv - running during an air raid,146.35,164.643750,48.267851



=== partner ===


,Title,Distance,speed_kmh,steps_per_km
46,Rimac Running,23.00,23.951403,338.260870
49,Lima Running,23.91,28.454876,269.259724



=== bob ===


,Title,Distance,speed_kmh,steps_per_km
64,San Martín de Porres - База,46.65,60.279971,121.414791
735,Без названия,34.56,41.499666,NaN



=== log ===


,Title,Distance,speed_kmh


#### Видалення знайдених аномалій:

In [17]:
# видалимо записи із наочним виведенням

datasets = {
    "df_personal": (df_personal, anomalies_personal),
    "df_partner": (df_partner, anomalies_partner),
    "df_activity_bob": (df_activity_bob, anomalies_bob),
    "df_activity_log": (df_activity_log, anomalies_log)
}

cleaned = {}
for name, (df, anomalies) in datasets.items():
    before = len(df)
    cleaned_df = df[~df.index.isin(anomalies.index)]
    print(f"{name}: {before} rows before, and {len(cleaned_df)} rows after cleaning, ({before - len(cleaned_df)} deleted)")
    cleaned[name] = cleaned_df

# 
df_personal = cleaned["df_personal"]
df_partner = cleaned["df_partner"]
df_activity_bob = cleaned["df_activity_bob"]
df_activity_log = cleaned["df_activity_log"]

df_personal: 299 rows before, and 295 rows after cleaning, (4 deleted)
df_partner: 278 rows before, and 276 rows after cleaning, (2 deleted)
df_activity_bob: 625 rows before, and 623 rows after cleaning, (2 deleted)
df_activity_log: 682 rows before, and 682 rows after cleaning, (0 deleted)


Ми опрацьовуємо дані спортсменів, які тренуються в Україні, під час війни. Знайдені вище атрефакти - це наслідок GPS-аномалій від роботи ППО при повітряних тривогах. Фактор війни присутній і в житті спортсменів. Окрім очевидного впливу, такі нюанси позбавляють змоги проаналізувати пройдену дистанцію та зроблені помилки. 

На перший погляд деякі з цих записів є нормальними швидкісними забігами.

Їх аномальний характер легко пропустити навіть зараз. Їх дивну природу слід додатково обгрунтувати і критерій швидкості руху виявляється недостатнім.

Врятувало таке спостереження - під час повітряних тривог GPS не працює і ламає Distance, але інші фізичні параметри годинник продовжує зчитувати сумлінно. Таким чином у нас лишається параметр Steps який правдиво відображає довжину пробіжки. Але виявилось це працює також не на всіх пристроях (а в df_activity_log взагалі відсутня колонка Steps - та це нам не заважає, оскільки там збережено дані американського бігуна і робота ППО не псує його датасет). 

Тож в якості критерія перевірки пробіжки на аномальність було використано steps_per_km на додачу до speed_kmh.


Порогові значення в функції обрано з двох міркувань:
- 20 км/год — фізична межа бігу людини (світовий рекорд ~23 км/год на короткій дистанції),
  тому темп вище 20 км/год в записі є GPS-помилкою.
- 500 крок/км — вдвічі нижче норми ходьби (~1000) і вчетверо нижче бігу (~1400),
  щоб не видалити повільні навігаційні відрізки, які є частиною нашого спорту.

Комбінація обох умов дає мінімальну кількість хибних записів.

Отже, ми виявили всі артефакти, один з них за 2011 рік, найперший запис в датасеті, мабуть тестовий, абсолютно нереалістичний (Best Pace = 0:02, NaN в більшості полів) - тому його теж видалено.

Ми завершили перевірку на аномальність і видалили усі підозрілі записи з 4 датасетів.
___

### КОНВЕРТАЦІЯ ДАНИХ

Date в усіх датасетах — це (object) і для часового аналізу треба її конвертувати в datetime

In [18]:
df_personal["Date"] = pd.to_datetime(df_personal["Date"])

df_partner["Date"] = pd.to_datetime(df_partner["Date"])

df_activity_bob["Date"] = pd.to_datetime(df_activity_bob["Date"])

df_activity_log["Date"] = pd.to_datetime(df_activity_log["Date"], format="mixed")

C:\Users\viver\AppData\Local\Temp\ipykernel_24208\1971355138.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_personal["Date"] = pd.to_datetime(df_personal["Date"])
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\1971355138.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_partner["Date"] = pd.to_datetime(df_partner["Date"])
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\1971355138.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

In [19]:
print(df_personal["Date"].head())
print(df_partner["Date"].head())
print(df_activity_bob["Date"].head())
print(df_activity_log["Date"].head())

0   2026-05-17 11:56:00
1   2026-05-16 14:20:24
2   2026-05-09 11:45:01
3   2026-05-03 10:16:03
4   2026-05-02 12:04:59
Name: Date, dtype: datetime64[ns]
0   2026-04-19 12:30:22
1   2026-04-19 11:45:37
2   2026-04-18 13:15:35
3   2026-04-14 16:56:03
4   2026-04-14 16:05:04
Name: Date, dtype: datetime64[ns]
0   2026-05-19 07:15:28
1   2026-05-16 10:30:56
2   2026-05-10 08:18:24
3   2026-05-09 06:22:41
4   2026-05-07 06:00:01
Name: Date, dtype: datetime64[ns]
0   2020-07-15 09:41:00
1   2020-07-14 17:45:00
2   2020-07-13 18:57:00
3   2020-07-12 18:44:00
4   2020-07-11 19:35:00
Name: Date, dtype: datetime64[ns]


Avg Pace та Best Pace слід конвертувати в число:

In [20]:
print(df_personal["Avg Pace"].head())
print(df_partner["Avg Pace"].head())
print(df_activity_bob["Avg Pace"].head())
print(df_activity_log["Avg Pace"].head())

0     7:58
1    11:24
2    13:54
3    11:01
4    10:07
Name: Avg Pace, dtype: object
0     9:47
1     8:25
2     7:24
3    14:49
4    10:33
Name: Avg Pace, dtype: object
0    11:51
1    11:52
2    10:26
3    10:49
4    10:50
Name: Avg Pace, dtype: object
0    7:19
1    7:14
2    8:05
3    7:33
4    8:01
Name: Avg Pace, dtype: object


In [21]:
def pace_to_float(pace_str):
    if pd.isna(pace_str) or pace_str is None:
        return None
    pace_str = str(pace_str).replace(',', '.')  # замінюємо кому на крапку
    if ":" in str(pace_str):
        parts = str(pace_str).split(":")
        return int(parts[0]) + int(parts[1]) / 60
    else:
        return float(pace_str)


df_personal["Avg Pace num"] = df_personal["Avg Pace"].apply(pace_to_float)

df_personal["Best Pace num"] = df_personal["Best Pace"].apply(pace_to_float)

print(df_personal["Avg Pace num"].head())
print(df_personal["Best Pace num"].head())

0     7.966667
1    11.400000
2    13.900000
3    11.016667
4    10.116667
Name: Avg Pace num, dtype: float64
0    5.233333
1    5.066667
2    4.633333
3    4.550000
4    4.316667
Name: Best Pace num, dtype: float64


C:\Users\viver\AppData\Local\Temp\ipykernel_24208\3385031731.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_personal["Avg Pace num"] = df_personal["Avg Pace"].apply(pace_to_float)
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\3385031731.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_personal["Best Pace num"] = df_personal["Best Pace"].apply(pace_to_float)


In [22]:
df_partner["Avg Pace num"] = df_partner["Avg Pace"].apply(pace_to_float)

df_partner["Best Pace num"] = df_partner["Best Pace"].apply(pace_to_float)

print(df_partner["Avg Pace num"].head())
print(df_partner["Best Pace num"].head())

0     9.783333
1     8.416667
2     7.400000
3    14.816667
4    10.550000
Name: Avg Pace num, dtype: float64
0    4.683333
1    5.000000
2    4.366667
3    6.633333
4    4.550000
Name: Best Pace num, dtype: float64


C:\Users\viver\AppData\Local\Temp\ipykernel_24208\68191573.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_partner["Avg Pace num"] = df_partner["Avg Pace"].apply(pace_to_float)
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\68191573.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_partner["Best Pace num"] = df_partner["Best Pace"].apply(pace_to_float)


In [23]:
df_activity_bob["Avg Pace num"] = df_activity_bob["Avg Pace"].apply(pace_to_float)

df_activity_bob["Best Pace num"] = df_activity_bob["Best Pace"].apply(pace_to_float)

print(df_activity_bob["Avg Pace num"].head())
print(df_activity_bob["Best Pace num"].head())

0    11.850000
1    11.866667
2    10.433333
3    10.816667
4    10.833333
Name: Avg Pace num, dtype: float64
0    6.816667
1    7.033333
2    6.133333
3    6.583333
4    6.516667
Name: Best Pace num, dtype: float64


C:\Users\viver\AppData\Local\Temp\ipykernel_24208\1554645275.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_activity_bob["Avg Pace num"] = df_activity_bob["Avg Pace"].apply(pace_to_float)
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\1554645275.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_activity_bob["Best Pace num"] = df_activity_bob["Best Pace"].apply(pace_to_float)


In [24]:
df_activity_log["Avg Pace num"] = df_activity_log["Avg Pace"].apply(pace_to_float)

df_activity_log["Best Pace num"] = df_activity_log["Best Pace"].apply(pace_to_float)

print(df_activity_log["Avg Pace num"].head())
print(df_activity_log["Best Pace num"].head())

0    7.316667
1    7.233333
2    8.083333
3    7.550000
4    8.016667
Name: Avg Pace num, dtype: float64
0    6.333333
1    6.583333
2    5.816667
3    5.000000
4    6.800000
Name: Best Pace num, dtype: float64


In [25]:
# перевірка

df_personal[["Avg Pace", "Avg Pace num", "Best Pace", "Best Pace num"]].head()

,Avg Pace,Avg Pace num,Best Pace,Best Pace num
0,7:58,7.966667,5:14,5.233333
1,11:24,11.400000,5:04,5.066667
2,13:54,13.900000,4:38,4.633333
3,11:01,11.016667,4:33,4.550000
4,10:07,10.116667,4:19,4.316667


___

### УНІФІКАЦІЯ НАЗВ КОЛОНОК

В df_activity маємо колонки Elev Gain і Elev Loss замість Total Ascent і Total Descent як в двох інших датасетах. Перейменовуємо:

In [26]:
df_activity_log = df_activity_log.rename(columns={"Elev Gain": "Total Ascent", "Elev Loss": "Total Descent"})

In [27]:
print(df_activity_log["Total Ascent"].head())
print(df_activity_log["Total Descent"].head())

0    169
1    183
2    124
3    215
4     76
Name: Total Ascent, dtype: object
0    173
1    187
2    124
3    219
4     80
Name: Total Descent, dtype: object


Скористаємось errors="coerce" — якщо лишились нечислові значення він поставить NaN:

In [28]:
for df in [df_personal, df_partner, df_activity_bob, df_activity_log]:
    df["Total Ascent"] = pd.to_numeric(df["Total Ascent"], errors="coerce")
    df["Total Descent"] = pd.to_numeric(df["Total Descent"], errors="coerce")

C:\Users\viver\AppData\Local\Temp\ipykernel_24208\3786910697.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Total Ascent"] = pd.to_numeric(df["Total Ascent"], errors="coerce")
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\3786910697.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Total Descent"] = pd.to_numeric(df["Total Descent"], errors="coerce")


In [29]:
for df in [df_personal, df_partner, df_activity_bob, df_activity_log]:
    df["Calories"] = pd.to_numeric(df["Calories"], errors="coerce")

C:\Users\viver\AppData\Local\Temp\ipykernel_24208\3186312824.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Calories"] = pd.to_numeric(df["Calories"], errors="coerce")


___

### ОБ'ЄДНУЄМО ДАНІ АТЛЕТІВ

створюємо додаткову колонку "Athlete" для ідентифікації (маркер):

In [30]:
df_personal["Athlete"] = "fox_1"

df_partner["Athlete"] = "fox_2"

df_activity_bob["Athlete"] = "not_a_fox_1"

df_activity_log["Athlete"] = "not_a_fox_2"

C:\Users\viver\AppData\Local\Temp\ipykernel_24208\2175827925.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_personal["Athlete"] = "fox_1"
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\2175827925.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_partner["Athlete"] = "fox_2"
C:\Users\viver\AppData\Local\Temp\ipykernel_24208\2175827925.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

комбінуємо датафрейми для подальшого аналізу:

In [31]:
df_combined = pd.concat([df_personal, df_partner, df_activity_bob, df_activity_log], ignore_index=True)

In [32]:
# перевірка

print(df_combined.shape)
print(df_combined["Athlete"].value_counts())

(1876, 47)
Athlete
not_a_fox_2    682
not_a_fox_1    623
fox_1          295
fox_2          276
Name: count, dtype: int64


Збережемо очищені дані:

In [33]:
df_combined.to_csv("../data/processed/garmin_combined.csv", index=False)

На цьому ми завершили процес знайомства з даними, їх очистку, конвертацію та об'єднаня в один датасет для подальшого аналізу
___

### EDA - > file 02_eda.ipynb